In [ ]:
"""Figures 3, 5 and 6 of the manuscript.

All values are copied from the notebook outputs and the reduced-feature
runs (Tables 3 and 4, Figure 5 importance scores, reduced-feature results).
"""

import numpy as np
import matplotlib.pyplot as plt

from corrosion_figstyle import (COLORS, WIDTH_DOUBLE, MM, apply_style,
                                panel_label, save)

apply_style()
MODELS = ["DT", "RF", "Bagging", "XGBoost"]


# ==================================================================
# FIGURE 3: baseline vs optimized accuracy (Tables 3 and 4)
# Grouped bars as in the submitted version, redrawn in the shared style.
# ==================================================================
FIG3 = {
    "Dataset No. 1": dict(
        base_train=[100.00, 100.00, 100.00, 99.94],
        base_test=[93.16, 88.93, 94.65, 95.27],
        opt_train=[100.00, 100.00, 100.00, 100.00],
        opt_test=[94.03, 94.53, 94.53, 95.52],
        opt_train_std=[0.00, 0.00, 0.00, 0.00],      # five-seed std
        opt_test_std=[1.08, 0.71, 0.73, 0.64],
    ),
    "Dataset No. 2": dict(
        base_train=[100.00, 100.00, 100.00, 100.00],
        base_test=[90.29, 75.73, 93.20, 94.50],
        opt_train=[98.06, 99.27, 100.00, 100.00],
        opt_test=[91.91, 94.17, 92.88, 95.47],
        opt_train_std=[0.19, 0.21, 0.00, 0.00],
        opt_test_std=[1.52, 1.13, 0.84, 0.83],
    ),
}
BARS = [  # key, std key, label, colour
    ("base_train", None, "Baseline train", "#D0D0D0"),
    ("base_test", None, "Baseline test", COLORS["baseline"]),
    ("opt_train", "opt_train_std", "Optimized train",
     COLORS["optimized_light"]),
    ("opt_test", "opt_test_std", "Optimized test", COLORS["optimized"]),
]

fig, axes = plt.subplots(1, 2, figsize=(WIDTH_DOUBLE, 66 * MM),
                         sharey=True, gridspec_kw={"wspace": 0.08})
x = np.arange(len(MODELS))
w = 0.2
for ax, (title, d), letter in zip(axes, FIG3.items(), "ab"):
    for i, (key, std_key, label, color) in enumerate(BARS):
        xi = x + (i - 1.5) * w
        yerr = d[std_key] if std_key else None
        ax.bar(xi, d[key], w, color=color, label=label, yerr=yerr,
               error_kw={"elinewidth": 1.0, "capthick": 1.0})
        top = np.array(d[key]) + (np.array(yerr) if yerr else 0)
        for xv, v, t in zip(xi, d[key], top):
            ax.text(xv, t + 1.5, f"{v:.2f}", rotation=90, ha="center",
                    va="bottom", fontsize=6, color=COLORS["text"])
    ax.set_xticks(x)
    ax.set_xticklabels(MODELS)
    ax.set_title(title, pad=4)
    panel_label(ax, letter, x=-0.12 if letter == "a" else -0.04)

axes[0].set_ylim(0, 122)
axes[0].set_yticks(range(0, 101, 20))
axes[0].set_ylabel("Accuracy (%)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4,
           bbox_to_anchor=(0.5, 0.99))
save(fig, "Figure3_accuracy", multipanel=True)
plt.close(fig)


# ==================================================================
# FIGURE 5: XGBoost feature importance (top 10), coloured by type
# ==================================================================
FIG5 = {
    "Dataset No. 1": [
        ("KNO$_3$", 0.099390, "environment"),
        ("KOH", 0.083289, "environment"),
        ("Fe (balance)", 0.079772, "composition"),
        ("KHSO$_4$", 0.074058, "environment"),
        ("HCl", 0.057552, "environment"),
        ("H$_2$SO$_4$", 0.055163, "environment"),
        ("Temperature", 0.047412, "environment"),
        ("Mo", 0.042788, "composition"),
        ("HNO$_3$", 0.042475, "environment"),
        ("NH$_4$Cl", 0.040446, "environment"),
    ],
    "Dataset No. 2": [
        ("[OH$^{-}$]", 0.298755, "environment"),
        ("[S$^{2-}$]", 0.150742, "environment"),
        ("Fe (balance)", 0.091960, "composition"),
        ("Mn", 0.074711, "composition"),
        ("Mo", 0.057763, "composition"),
        ("[H$^{+}$]", 0.047910, "environment"),
        ("[Cl$^{-}$]", 0.047053, "environment"),
        ("S", 0.039581, "composition"),
        ("P", 0.036266, "composition"),
        ("Al", 0.028739, "composition"),
    ],
}

fig, axes = plt.subplots(1, 2, figsize=(WIDTH_DOUBLE, 62 * MM),
                         gridspec_kw={"wspace": 0.42})
for ax, (title, rows), letter in zip(axes, FIG5.items(), "ab"):
    names = [r[0] for r in rows][::-1]
    vals = [r[1] for r in rows][::-1]
    cols = [COLORS[r[2]] for r in rows][::-1]
    ax.barh(names, vals, color=cols, height=0.68)
    ax.set_xlabel("Feature importance")
    ax.set_title(title, pad=4)
    # 7.5 pt so that mathtext superscripts stay above the 5 pt floor
    ax.tick_params(axis="y", length=0, labelsize=7.5)
    ax.set_xlim(0, max(vals) * 1.08)
    panel_label(ax, letter, x=-0.30)

from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(color=COLORS["environment"],
                              label="Environmental descriptor"),
                        Patch(color=COLORS["composition"],
                              label="Alloy composition")],
               loc="lower right")
save(fig, "Figure5_feature_importance", multipanel=True)
plt.close(fig)


# ==================================================================
# FIGURE 6: XGBoost with reduced descriptor sets
# ==================================================================
FIG6 = {
    "Dataset No. 1": dict(n_all=28,
                          locked=[52.49, 87.06, 93.53, 95.52],
                          mean=[52.49, 87.64, 92.96, 95.90],
                          std=[1.07, 0.87, 1.02, 0.64]),
    "Dataset No. 2": dict(n_all=17,
                          locked=[80.91, 93.20, 95.47, 95.47],
                          mean=[82.33, 96.25, 96.44, 96.44],
                          std=[7.04, 0.96, 0.94, 0.83]),
}

fig, axes = plt.subplots(1, 2, figsize=(WIDTH_DOUBLE, 60 * MM),
                         sharey=True, gridspec_kw={"wspace": 0.08})
w = 0.36
for ax, (title, d), letter in zip(axes, FIG6.items(), "ab"):
    xx = np.arange(4)
    ax.bar(xx - w / 2, d["locked"], w, color=COLORS["optimized"],
           label="Locked test set")
    ax.bar(xx + w / 2, d["mean"], w, yerr=d["std"],
           color=COLORS["optimized_light"],
           error_kw={"elinewidth": 1.0, "capthick": 1.0},
           label="Five-seed mean ± std")
    for xi, v in zip(xx - w / 2, d["locked"]):
        ax.text(xi, v + 1.2, f"{v:.2f}", ha="center", va="bottom",
                fontsize=6.5, color=COLORS["optimized"])
    ax.set_xticks(xx)
    ax.set_xticklabels(["Top 5", "Top 10", "Top 15",
                        f"All ({d['n_all']})"])
    ax.set_xlabel("Descriptors used")
    ax.set_title(title, pad=4)
    panel_label(ax, letter, x=-0.12 if letter == "a" else -0.04)

axes[0].set_ylim(0, 105)
axes[0].set_ylabel("Test accuracy (%)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2,
           bbox_to_anchor=(0.5, 0.99))
save(fig, "Figure6_reduced_features", multipanel=True)
plt.close(fig)

In [6]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()  # repository root
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
"""Figures 4, 7, 8 and 9 of the manuscript, redrawn from saved results.

Inputs (paths can be changed below):
  Figure 4  roc_dataset1.csv, roc_dataset2.csv
            (written by export_roc_data.py in each notebook)
  Figure 7  results/multiseed_learning_curve_summary.csv
            (written by analyze_multiseed.py)
  Figure 8  results/active_classification/multiseed_active_iteration_summary.csv
  Figure 9  results/active_classification/multiseed_class_distribution_summary.csv
            (both written by analyze_active_classification.py)

Each figure is skipped with a message if its input file is missing.
"""

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

from corrosion_figstyle import (COLORS, CLASS_ORDER, SURROGATE_STYLE,
                                MM, WIDTH_ONEHALF, apply_style,
                                panel_label, save)

apply_style()

ROC_FILES = {"Dataset No. 1": ROOT / "results" / "roc_dataset1.csv",
             "Dataset No. 2": ROOT / "results" / "roc_dataset2.csv"}
BO_CURVES = ROOT / "results" / "multiseed_learning_curve_summary.csv"
RCR_CURVES = ROOT / "results" / "active_classification" / "multiseed_active_iteration_summary.csv"
CLASS_DIST = ROOT / "results" / "active_classification" / "multiseed_class_distribution_summary.csv"


def missing(path):
    if not Path(path).exists():
        print(f"Skipped: {path} not found")
        return True
    return False


# ==================================================================
# FIGURE 4: one-vs-rest ROC curves of the optimized XGBoost models
# ==================================================================
if not any(missing(p) for p in ROC_FILES.values()):
    fig, axes = plt.subplots(1, 2, figsize=(WIDTH_ONEHALF, 72 * MM),
                             gridspec_kw={"wspace": 0.12})
    for ax, (title, path), letter in zip(axes, ROC_FILES.items(), "ab"):
        df = pd.read_csv(path)
        y = df["true_class"].astype(str)
        aucs = []
        for cls in CLASS_ORDER:
            score = df[f"prob_{cls}"]
            truth = (y == cls).astype(int)
            fpr, tpr, _ = roc_curve(truth, score)
            auc = roc_auc_score(truth, score)
            aucs.append(auc)
            ax.plot(fpr, tpr, color=COLORS[cls], lw=1.6,
                    label=f"{cls} ({auc:.3f})")
        ax.plot([0, 1], [0, 1], ls=":", lw=1.0, color=COLORS["Random"])
        ax.set_xlim(-0.01, 1.0)
        ax.set_ylim(0, 1.01)
        ax.set_aspect("equal")
        ax.set_xlabel("False positive rate")
        ax.set_title(f"{title} (macro AUC = {np.mean(aucs):.3f})", pad=4)
        ax.legend(loc="lower right", title="Class (AUC)",
                  title_fontsize=7, handlelength=1.4)
        panel_label(ax, letter, x=-0.2 if letter == "a" else -0.08)
    axes[0].set_ylabel("True positive rate")
    axes[1].tick_params(labelleft=False)
    save(fig, "Figure4_ROC", multipanel=True)
    plt.close(fig)


# ==================================================================
# FIGURE 7: BO learning curves (colour = acquisition, line = surrogate)
# ==================================================================
if not missing(BO_CURVES):
    df = pd.read_csv(BO_CURVES)
    fig, ax = plt.subplots(figsize=(WIDTH_ONEHALF, 82 * MM))
    for surrogate in ["RF", "GP"]:
        for acq in ["EI", "PI", "UCB"]:
            d = df[(df["surrogate"] == surrogate)
                   & (df["acquisition"] == acq)].sort_values("train_size")
            if d.empty:
                continue
            x = d["train_size"].to_numpy()
            m = d["R2_mean"].to_numpy()
            s = d["R2_std"].fillna(0).to_numpy()
            ax.fill_between(x, m - s, m + s, color=COLORS[acq],
                            alpha=0.10, lw=0)
            ax.plot(x, m, color=COLORS[acq], ls=SURROGATE_STYLE[surrogate],
                    lw=1.6, label=f"{surrogate}-{acq}")
    ax.set_xlabel("Training-set size")
    ax.set_ylabel("Mean $R^2$ (held-out test set)")
    ax.set_xlim(0, 3500)
    ax.set_ylim(-0.35, 1.02)
    ax.legend(loc="lower right", ncol=2, handlelength=2.6,
              columnspacing=1.2)
    save(fig, "Figure7_BO_learning_curves")
    plt.close(fig)


# ==================================================================
# FIGURE 8: resistant class ratio during active classification
# ==================================================================
if not missing(RCR_CURVES):
    df = pd.read_csv(RCR_CURVES)
    fig, ax = plt.subplots(figsize=(WIDTH_ONEHALF, 78 * MM))
    for strategy in ["PI", "EI", "UCB", "Random"]:
        d = df[df["strategy"] == strategy].sort_values("iteration")
        x = d["iteration"].to_numpy()
        m = d["rcr_mean"].to_numpy()
        s = d["rcr_std"].fillna(0).to_numpy()
        ax.fill_between(x, m - s, m + s, color=COLORS[strategy],
                        alpha=0.15, lw=0)
        ax.plot(x, m, color=COLORS[strategy], lw=1.8, label=strategy)
    ax.set_xlabel("Acquisition iteration")
    ax.set_ylabel("Resistant class ratio")
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 0.8)                       # 0.8 fully visible
    ax.set_yticks(np.arange(0, 0.81, 0.2))
    ax.legend(loc="center right")
    save(fig, "Figure8_RCR")
    plt.close(fig)


# ==================================================================
# FIGURE 9: class composition of the training set (PI)
# ==================================================================
if not missing(CLASS_DIST):
    df = pd.read_csv(CLASS_DIST)
    d = df[df["strategy"] == "PI"].pivot(index="iteration",
                                         columns="class_name",
                                         values="count_mean")[CLASS_ORDER]
    fig, ax = plt.subplots(figsize=(WIDTH_ONEHALF, 78 * MM))
    ax.stackplot(d.index, *[d[c].to_numpy() for c in CLASS_ORDER],
                 labels=CLASS_ORDER,
                 colors=[COLORS[c] for c in CLASS_ORDER],
                 edgecolor="white", linewidth=0.4)
    ax.set_xlabel("Acquisition iteration")
    ax.set_ylabel("Mean number of samples")
    ax.set_xlim(d.index.min(), d.index.max())
    ax.set_ylim(0, 1150)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], labels[::-1], loc="upper left")
    save(fig, "Figure9_class_distribution_PI")
    plt.close(fig)

  (alignment audit not available; skipped)


Saved figures_publication\Figure4_ROC.[svg|pdf|tiff|png]
Saved figures_publication\Figure7_BO_learning_curves.[svg|pdf|tiff|png]
Saved figures_publication\Figure8_RCR.[svg|pdf|tiff|png]
Saved figures_publication\Figure9_class_distribution_PI.[svg|pdf|tiff|png]


In [2]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()  # repository root
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
"""Supervisor question: is one environmental condition tested on many
different alloy compositions with different corrosion behaviour?

If so, the environment can be fixed and BO run over alloy compositions only.

Usage (next to the CSV files used by the notebooks):
    python check_environment_groups.py "Dataset No.1.csv"
    python check_environment_groups.py "Dataset No.2.csv"

Writes <csv name>_environment_groups.csv and prints a summary.
"""

import sys
from pathlib import Path

import pandas as pd

# ------------------------------------------------------------
# Dataset file. In a notebook, just edit this line.
# From a terminal, the file can also be given as an argument.
# ------------------------------------------------------------
CSV = ROOT / "data" / "Dataset No.1.csv"

args = [a for a in sys.argv[1:] if a.lower().endswith(".csv")]
if args:                       # terminal: python check_environment_groups.py file.csv
    CSV = args[0]
CSV = Path(CSV)                # works whether CSV is a string or a Path

ELEMENTS = {"C", "Mn", "Si", "P", "S", "Cr", "Mo", "Ni", "N", "Ti", "Nb",
            "Al"}
NON_FEATURES = {"ASTM Material", "Behavior", "Pitting_label"}

df = pd.read_csv(CSV)
df.columns = df.columns.str.strip()

comp_cols = [c for c in df.columns
             if c in ELEMENTS or c.lower().startswith("fe")]
env_cols = [c for c in df.columns
            if c not in comp_cols and c not in NON_FEATURES
            and pd.api.types.is_numeric_dtype(df[c])]
env_cols = [c for c in env_cols if df[c].nunique() > 1]   # drop constants

print(f"{CSV.name}: {len(df)} rows")
print(f"  composition columns ({len(comp_cols)}): {comp_cols}")
print(f"  environment columns ({len(env_cols)}): {env_cols}")

# One environment = one unique combination of all environment columns.
env_key = df[env_cols].round(6).astype(str).agg("|".join, axis=1)
df["environment_id"] = pd.factorize(env_key)[0]
df["composition_id"] = pd.factorize(
    df[comp_cols].round(6).astype(str).agg("|".join, axis=1))[0]

rows = []
for env_id, g in df.groupby("environment_id"):
    counts = g["Behavior"].value_counts()
    first = g.iloc[0]
    rows.append({
        "environment_id": env_id,
        "condition": "; ".join(f"{c}={first[c]:g}" for c in env_cols
                               if first[c] != 0),
        "n_samples": len(g),
        "n_alloys": g["ASTM Material"].nunique(),
        "n_compositions": g["composition_id"].nunique(),
        "n_classes": g["Behavior"].nunique(),
        **{f"n_{k}": int(counts.get(k, 0))
           for k in ["Severe", "Poor", "Good", "Resistant"]},
    })
groups = pd.DataFrame(rows).sort_values(
    ["n_compositions", "n_classes"], ascending=False)

out = CSV.with_name(CSV.stem.replace(" ", "_") + "_environment_groups.csv")
groups.to_csv(out, index=False)

mixed = groups[(groups["n_classes"] >= 2)]
contrast = groups[(groups["n_Resistant"] > 0)
                  & ((groups["n_Severe"] + groups["n_Poor"]) > 0)]

print(f"\nUnique environments: {len(groups)}")
print("Alloy compositions per environment: "
      f"median {groups['n_compositions'].median():.0f}, "
      f"max {groups['n_compositions'].max()}")
for k in [5, 10, 20]:
    n_k = (groups["n_compositions"] >= k).sum()
    n_k_mixed = ((groups["n_compositions"] >= k)
                 & (groups["n_classes"] >= 2)).sum()
    print(f"  environments with >= {k:2d} compositions: {n_k:4d} "
          f"(of which {n_k_mixed} show >= 2 behaviour classes)")
print(f"Environments with >= 2 behaviour classes: {len(mixed)}")
print("Environments containing both Resistant and Severe/Poor alloys: "
      f"{len(contrast)} ({contrast['n_samples'].sum()} samples)")
print("\nTop 10 environments by number of compositions:")
print(groups.head(10).to_string(index=False))
print(f"\nSaved {out}")

Dataset No.1.csv: 4019 rows
  composition columns (13): ['C', 'Mn', 'Si', 'P', 'S', 'Cr', 'Mo', 'Ni', 'N', 'Ti', 'Nb', 'Al', 'Fe balance (%)']
  environment columns (15): ['Formic acid (HCOOH)', 'Ammonium chloride(NH4Cl)', 'Acetic acid(CH3COOH)', 'Potasium Hydroxide (KOH)', 'Lactic acid', 'oxalic acid (COOH)2.2H2O', 'Phosphric acid (H3PO4)', 'Sulfuric acid (H2SO4)', 'Nitric acid (HNO3)', 'Hydrocloric acid (HCl)', 'Citric acid(HOC(CH2COOH)2COOH.H2O', 'KHSO4', 'KNO3', 'MgCl2.6H2O', 'Temperature']

Unique environments: 122
Alloy compositions per environment: median 30, max 30
  environments with >=  5 compositions:  122 (of which 96 show >= 2 behaviour classes)
  environments with >= 10 compositions:  122 (of which 96 show >= 2 behaviour classes)
  environments with >= 20 compositions:  117 (of which 91 show >= 2 behaviour classes)
Environments with >= 2 behaviour classes: 96
Environments containing both Resistant and Severe/Poor alloys: 43 (1452 samples)

Top 10 environments by number of

In [1]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()  # repository root
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
"""Supervisor question: is one environmental condition tested on many
different alloy compositions with different corrosion behaviour?

If so, the environment can be fixed and BO run over alloy compositions only.

Usage (next to the CSV files used by the notebooks):
    python check_environment_groups.py "Dataset No.1.csv"
    python check_environment_groups.py "Dataset No.2.csv"

Writes <csv name>_environment_groups.csv and prints a summary.
"""

import sys
from pathlib import Path

import pandas as pd

# ------------------------------------------------------------
# Dataset file. In a notebook, just edit this line.
# From a terminal, the file can also be given as an argument.
# ------------------------------------------------------------
CSV = ROOT / "data" / "Dataset No.2.csv"

args = [a for a in sys.argv[1:] if a.lower().endswith(".csv")]
if args:                       # terminal: python check_environment_groups.py file.csv
    CSV = args[0]
CSV = Path(CSV)                # works whether CSV is a string or a Path

ELEMENTS = {"C", "Mn", "Si", "P", "S", "Cr", "Mo", "Ni", "N", "Ti", "Nb",
            "Al"}
NON_FEATURES = {"ASTM Material", "Behavior", "Pitting_label"}

df = pd.read_csv(CSV)
df.columns = df.columns.str.strip()

comp_cols = [c for c in df.columns
             if c in ELEMENTS or c.lower().startswith("fe")]
env_cols = [c for c in df.columns
            if c not in comp_cols and c not in NON_FEATURES
            and pd.api.types.is_numeric_dtype(df[c])]
env_cols = [c for c in env_cols if df[c].nunique() > 1]   # drop constants

print(f"{CSV.name}: {len(df)} rows")
print(f"  composition columns ({len(comp_cols)}): {comp_cols}")
print(f"  environment columns ({len(env_cols)}): {env_cols}")

# One environment = one unique combination of all environment columns.
env_key = df[env_cols].round(6).astype(str).agg("|".join, axis=1)
df["environment_id"] = pd.factorize(env_key)[0]
df["composition_id"] = pd.factorize(
    df[comp_cols].round(6).astype(str).agg("|".join, axis=1))[0]

rows = []
for env_id, g in df.groupby("environment_id"):
    counts = g["Behavior"].value_counts()
    first = g.iloc[0]
    rows.append({
        "environment_id": env_id,
        "condition": "; ".join(f"{c}={first[c]:g}" for c in env_cols
                               if first[c] != 0),
        "n_samples": len(g),
        "n_alloys": g["ASTM Material"].nunique(),
        "n_compositions": g["composition_id"].nunique(),
        "n_classes": g["Behavior"].nunique(),
        **{f"n_{k}": int(counts.get(k, 0))
           for k in ["Severe", "Poor", "Good", "Resistant"]},
    })
groups = pd.DataFrame(rows).sort_values(
    ["n_compositions", "n_classes"], ascending=False)

out = CSV.with_name(CSV.stem.replace(" ", "_") + "_environment_groups.csv")
groups.to_csv(out, index=False)

mixed = groups[(groups["n_classes"] >= 2)]
contrast = groups[(groups["n_Resistant"] > 0)
                  & ((groups["n_Severe"] + groups["n_Poor"]) > 0)]

print(f"\nUnique environments: {len(groups)}")
print("Alloy compositions per environment: "
      f"median {groups['n_compositions'].median():.0f}, "
      f"max {groups['n_compositions'].max()}")
for k in [5, 10, 20]:
    n_k = (groups["n_compositions"] >= k).sum()
    n_k_mixed = ((groups["n_compositions"] >= k)
                 & (groups["n_classes"] >= 2)).sum()
    print(f"  environments with >= {k:2d} compositions: {n_k:4d} "
          f"(of which {n_k_mixed} show >= 2 behaviour classes)")
print(f"Environments with >= 2 behaviour classes: {len(mixed)}")
print("Environments containing both Resistant and Severe/Poor alloys: "
      f"{len(contrast)} ({contrast['n_samples'].sum()} samples)")
print("\nTop 10 environments by number of compositions:")
print(groups.head(10).to_string(index=False))
print(f"\nSaved {out}")

Dataset No.2.csv: 1544 rows
  composition columns (13): ['C', 'Mn', 'Si', 'P', 'S', 'Cr', 'Mo', 'Ni', 'N', 'Ti', 'Nb', 'Al', 'Fe balance (%)']
  environment columns (4): ['[H+] (mol/dm3)', '[OH-] (mol/dm3)', '[Cl-] (mol/dm3)', '[S2-] (mol/dm3)']

Unique environments: 47
Alloy compositions per environment: median 30, max 30
  environments with >=  5 compositions:   47 (of which 33 show >= 2 behaviour classes)
  environments with >= 10 compositions:   47 (of which 33 show >= 2 behaviour classes)
  environments with >= 20 compositions:   46 (of which 32 show >= 2 behaviour classes)
Environments with >= 2 behaviour classes: 33
Environments containing both Resistant and Severe/Poor alloys: 20 (670 samples)

Top 10 environments by number of compositions:
 environment_id                                     condition  n_samples  n_alloys  n_compositions  n_classes  n_Severe  n_Poor  n_Good  n_Resistant
             27 [H+] (mol/dm3)=0.3732; [S2-] (mol/dm3)=0.1866         34        34          

In [1]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()  # repository root
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
"""Figures 4, 7, 8 and 9 of the manuscript, redrawn from saved results.

Inputs (paths can be changed below):
  Figure 4  roc_dataset1.csv, roc_dataset2.csv
            (written by export_roc_data.py in each notebook)
  Figure 7  results/multiseed_learning_curve_summary.csv
            (written by analyze_multiseed.py)
  Figure 8  results/active_classification/multiseed_active_iteration_summary.csv
  Figure 9  results/active_classification/multiseed_class_distribution_summary.csv
            (both written by analyze_active_classification.py)

Each figure is skipped with a message if its input file is missing.
"""

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

from corrosion_figstyle import (COLORS, CLASS_ORDER, SURROGATE_STYLE,
                                BAND_ALPHA, MM, WIDTH_ONEHALF,
                                WIDTH_DOUBLE, apply_style, panel_label,
                                save)
apply_style()

ROC_FILES = {"Dataset No. 1": ROOT / "results" / "roc_dataset1.csv",
             "Dataset No. 2": ROOT / "results" / "roc_dataset2.csv"}
BO_CURVES = ROOT / "results" / "multiseed_learning_curve_summary.csv"
RCR_CURVES = ROOT / "results" / "active_classification" / "multiseed_active_iteration_summary.csv"
CLASS_DIST = ROOT / "results" / "active_classification" / "multiseed_class_distribution_summary.csv"


def missing(path):
    if not Path(path).exists():
        print(f"Skipped: {path} not found")
        return True
    return False

# ==================================================================
# FIGURE 4: one-vs-rest ROC curves of the optimized XGBoost models
# Axes are zoomed to the upper-left corner, where all curves lie
# (all curves exceed TPR = 0.7 at FPR = 0 and reach TPR = 1 before
# FPR = 0.25 in both datasets).
# ==================================================================
FPR_MAX, TPR_MIN = 0.3, 0.6
if not any(missing(p) for p in ROC_FILES.values()):
    fig, axes = plt.subplots(1, 2, figsize=(WIDTH_DOUBLE, 78 * MM),
                             sharey=True, gridspec_kw={"wspace": 0.08})
    for ax, (title, path), letter in zip(axes, ROC_FILES.items(), "ab"):
        df = pd.read_csv(path)
        y = df["true_class"].astype(str)
        aucs = []
        for cls in CLASS_ORDER:
            score = df[f"prob_{cls}"]
            truth = (y == cls).astype(int)
            fpr, tpr, _ = roc_curve(truth, score)
            auc = roc_auc_score(truth, score)
            aucs.append(auc)
            # trim to the zoom window (identical picture, no hidden
            # path segments running under the tick labels)
            fpr = np.clip(fpr, None, FPR_MAX)
            tpr = np.clip(tpr, TPR_MIN, None)
            ax.plot(fpr, tpr, color=COLORS[cls], lw=1.8,
                    label=f"{cls} ({auc:.3f})")
        ax.set_xlim(-0.003, FPR_MAX)
        ax.set_ylim(TPR_MIN, 1.005)
        ax.set_xlabel("False positive rate")
        ax.set_title(f"{title} (macro AUC = {np.mean(aucs):.3f})", pad=4)
        ax.legend(loc="lower right", title="Class (AUC)",
                  title_fontsize=7, handlelength=1.6)
        panel_label(ax, letter, x=-0.12 if letter == "a" else -0.04)
    axes[0].set_ylabel("True positive rate")
    save(fig, "Figure4_ROC", multipanel=True)
    plt.close(fig)


# ==================================================================
# FIGURE 7: BO learning curves (colour = acquisition, line = surrogate)
# ==================================================================
if not missing(BO_CURVES):
    df = pd.read_csv(BO_CURVES)
    fig, ax = plt.subplots(figsize=(WIDTH_ONEHALF, 82 * MM))
    for surrogate in ["RF", "GP"]:
        for acq in ["EI", "PI", "UCB"]:
            d = df[(df["surrogate"] == surrogate)
                   & (df["acquisition"] == acq)].sort_values("train_size")
            if d.empty:
                continue
            x = d["train_size"].to_numpy()
            m = d["R2_mean"].to_numpy()
            s = d["R2_std"].fillna(0).to_numpy()
            ax.fill_between(x, m - s, m + s, color=COLORS[acq],
                            alpha=BAND_ALPHA if surrogate == "RF"
                            else BAND_ALPHA * 0.6, lw=0)
            ax.plot(x, m, color=COLORS[acq], ls=SURROGATE_STYLE[surrogate],
                    lw=1.6, label=f"{surrogate}-{acq}")
    ax.set_xlabel("Training-set size")
    ax.set_ylabel("Mean $R^2$ (held-out test set)")
    ax.set_xlim(0, 3500)
    ax.set_ylim(-0.35, 1.02)
    ax.legend(loc="lower right", ncol=2, handlelength=2.6,
              columnspacing=1.2)
    save(fig, "Figure7_BO_learning_curves")
    plt.close(fig)


# ==================================================================
# FIGURE 8: resistant class ratio during active classification
# ==================================================================
if not missing(RCR_CURVES):
    df = pd.read_csv(RCR_CURVES)
    fig, ax = plt.subplots(figsize=(WIDTH_ONEHALF, 78 * MM))
    for strategy in ["PI", "EI", "UCB", "Random"]:
        d = df[df["strategy"] == strategy].sort_values("iteration")
        x = d["iteration"].to_numpy()
        m = d["rcr_mean"].to_numpy()
        s = d["rcr_std"].fillna(0).to_numpy()
        ax.fill_between(x, m - s, m + s, color=COLORS[strategy],
                        alpha=BAND_ALPHA, lw=0)
        ax.plot(x, m, color=COLORS[strategy], lw=1.8, label=strategy)
    ax.set_xlabel("Acquisition iteration")
    ax.set_ylabel("Resistant class ratio")
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 0.8)                       # 0.8 fully visible
    ax.set_yticks(np.arange(0, 0.81, 0.2))
    ax.legend(loc="center right")
    save(fig, "Figure8_RCR")
    plt.close(fig)


# ==================================================================
# FIGURE 9: class composition of the training set (PI)
# ==================================================================
if not missing(CLASS_DIST):
    df = pd.read_csv(CLASS_DIST)
    d = df[df["strategy"] == "PI"].pivot(index="iteration",
                                         columns="class_name",
                                         values="count_mean")[CLASS_ORDER]
    fig, ax = plt.subplots(figsize=(WIDTH_ONEHALF, 78 * MM))
    ax.stackplot(d.index, *[d[c].to_numpy() for c in CLASS_ORDER],
                 labels=CLASS_ORDER,
                 colors=[COLORS[c] for c in CLASS_ORDER],
                 edgecolor="white", linewidth=0.4)
    ax.set_xlabel("Acquisition iteration")
    ax.set_ylabel("Mean number of samples")
    ax.set_xlim(d.index.min(), d.index.max())
    ax.set_ylim(0, 1150)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], labels[::-1], loc="upper left")
    save(fig, "Figure9_class_distribution_PI")
    plt.close(fig)

  (alignment audit not available; skipped)


Saved figures_publication\Figure4_ROC.[svg|pdf|tiff|png]
Saved figures_publication\Figure7_BO_learning_curves.[svg|pdf|tiff|png]
Saved figures_publication\Figure8_RCR.[svg|pdf|tiff|png]
Saved figures_publication\Figure9_class_distribution_PI.[svg|pdf|tiff|png]
